# Stage 00b — Descriptions CSV + Figure Cropping & Label Matching

**What this notebook does:**

1. Exports all *Brief Description of the Drawings* texts from the PatSeer Excel
   into a single `data/descriptions.csv`.

2. Detects and crops every sub-figure on each drawing sheet using
   **DocLayout-YOLO** (document layout detector) + **EasyOCR** (label reader) —
   free, local, GPU-accelerated.

The matching logic:
1. DocLayout-YOLO detects `figure` and `figure_caption` regions on each sheet
2. For each figure box, EasyOCR tries to read the label via a cascade of passes:
   top-left corner → top-right corner → below strip (350 px) → side margins
3. Each crop is saved as `_F<label>.png` (matched) or `_Fu.png` (needs review)
4. FAT files are always copied whole as `_Fu`

**Output naming:**

| Label source | Output filename | Meaning |
|---|---|---|
| OCR label match | `US…_F1A.png` | matched to FIG. 1A |
| Unmatched / FAT / error | `US…_Fu.png` | needs review |

Output is written to `matched/<patent_id>/` — raw files are never modified.

**Where it fits in the pipeline:**
```
00a  (PatSeer download)
 ↓
00b  ←  YOU ARE HERE  (descriptions CSV + DocLayout-YOLO → _F / _Fu labels)
 ↓
01   (human review for _Fu files)
 ↓
02   (pad + resize to 518×518)
```

---

| Cell | What it does |
|------|------|
| 1 | CUDA check |
| 2 | Load config + Excel, save descriptions.csv |
| 3 | Build DocLayout-YOLO + EasyOCR engine (once per session) |
| 4 | Run pipeline over all patents (respects `scan_limit` from config) |
| 5 | Save review CSV + show flagged crops |

In [1]:
import torch
print("Is CUDA active inside Jupyter?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Connected GPU Model:", torch.cuda.get_device_name(0))

Is CUDA active inside Jupyter?: True
Connected GPU Model: NVIDIA GeForce RTX 2080 Ti


In [ ]:
import sys
import os
import pandas as pd
from pathlib import Path

notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent
src_dir = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from config_loader import load_config
from extractor import load_patseer_excel, save_descriptions_csv
import doclayout_matcher as dm

cfg = load_config()

print("Config loaded. Key paths:")
print("  raw_images :", cfg["paths"]["raw_images"])
print("  matched    :", cfg["paths"]["matched"])
print("  data       :", cfg["paths"]["data"])

In [ ]:
excel_path = Path(cfg["paths"]["patseer_excel"])
print(f"Reading PatSeer data from: {excel_path.name}...")
patseer_index = load_patseer_excel(excel_path)

desc_csv_path = Path(cfg["paths"]["data"]) / "descriptions.csv"
save_descriptions_csv(patseer_index, desc_csv_path)

df = pd.read_csv(desc_csv_path)
print(f"descriptions.csv saved — {len(df)} records.")

In [ ]:
# Build DocLayout-YOLO + EasyOCR engine — load once, reuse for all patents.
# First run downloads EasyOCR models (~100 MB). Subsequent runs are instant.
engine = dm.build_engine()
model, reader, device = engine
print("Engine ready on:", device)
print("YOLO classes   :", model.names)

In [ ]:
import re, time, shutil

raw_dir     = Path(cfg["paths"]["raw_images"])
matched_dir = Path(cfg["paths"]["matched"])
matched_dir.mkdir(parents=True, exist_ok=True)

scan_limit = cfg["extractor"].get("scan_limit", None)
run_df     = df.head(scan_limit) if scan_limit else df
print(f"Running on {len(run_df)} patents  (scan_limit={scan_limit})...")

# Normalise a patent ID to its bare alphanumeric core for folder matching.
# Strips download-tool suffixes (PAFP, PAF) then kind suffixes (A1, B1, …).
_CLEAN_RE   = re.compile(r"[^A-Za-z0-9]")
_DL_SUFFIX  = re.compile(r"PAFP$|PAF$", re.IGNORECASE)
_KIND_CODES = ["A1","A2","A3","B1","B2","C1","U1"]

def _core(pid: str) -> str:
    c = _CLEAN_RE.sub("", pid).upper()
    c = _DL_SUFFIX.sub("", c)          # strip PAFP / PAF first
    for sfx in _KIND_CODES:
        if c.endswith(sfx):
            return c[:-len(sfx)]
    return c

folder_map = {_core(p.name): p for p in raw_dir.iterdir() if p.is_dir()}

# Drawing sheet filename patterns across all 1639 patents:
#   img0001.png  imgf0001.png  imgaf001.png  fig_04.png  record__fig_04.png
_SHEET_RE = re.compile(
    r"^(?:img(?:af?)?\d|fig_\d|record__fig_\d)",
    re.IGNORECASE
)

rows = []
t0 = time.time()

for _, row in run_df.iterrows():
    excel_id = str(row.get("patent_id", "")).strip()
    if not excel_id:
        continue

    folder = folder_map.get(_core(excel_id))
    if folder is None:
        print(f"  ⚠  No raw folder for {excel_id} — skipping")
        continue

    out_dir = matched_dir / folder.name
    out_dir.mkdir(parents=True, exist_ok=True)

    files     = sorted(folder.iterdir())
    img_files = [f for f in files if _SHEET_RE.match(f.name) and f.suffix.lower() == ".png"]
    fat_files = [f for f in files if re.search(r"_FAT\d", f.name)]

    for f in fat_files:
        out_path = out_dir / f"{f.stem}_Fu.png"
        shutil.copy2(f, out_path)
        rows.append({"patent_id": excel_id, "original": f.name, "output": out_path.name,
                     "label": None, "method": "fat_copy", "needs_review": True})

    for img_path in img_files:
        try:
            res = dm.process_image(engine, img_path, out_dir)
            for c in res["crops"]:
                rows.append({"patent_id": excel_id, "original": c["original"],
                             "output": c["output"], "label": c["label"],
                             "method": c["method"], "needs_review": c["needs_review"]})
        except Exception as e:
            print(f"    ❌ {img_path.name}: {e}")

    labelled = sum(1 for r in rows if r["patent_id"] == excel_id and not r["needs_review"])
    total    = sum(1 for r in rows if r["patent_id"] == excel_id)
    print(f"  ✓ {excel_id}  sheets={len(img_files)}  crops={total}  labelled={labelled}")

elapsed = time.time() - t0
results_df = pd.DataFrame(rows)

crops_csv = Path(cfg["paths"]["data"]) / "crops_mapping.csv"
results_df.to_csv(crops_csv, index=False)

print(f"\nDone in {elapsed:.0f}s  ({elapsed/max(1,len(run_df)):.1f}s/patent)")
print(f"Total crops : {len(results_df)}")
if not results_df.empty:
    labelled_n = (results_df["needs_review"] == False).sum()
    print(f"Auto-labelled: {labelled_n}/{len(results_df)} = {100*labelled_n/len(results_df):.0f}%")
    print(results_df.groupby("method")["output"].count())

In [ ]:
if results_df.empty or "needs_review" not in results_df.columns:
    print("No crops produced — nothing to review.")
else:
    flagged_df = results_df[results_df["needs_review"] == True]
    review_csv = Path(cfg["paths"]["data"]) / "needs_human_review.csv"
    flagged_df.to_csv(review_csv, index=False)
    print(f"Saved {len(flagged_df)} flagged crops → {review_csv.name}")
    print(f"(Pass these to Step 01 — human review notebook)")
    if not flagged_df.empty:
        display(flagged_df[["patent_id","original","output","label","method"]].head(20))